# MechFaithBench: закрашивание области и контроль (эталон R0)

Ноутбук для Kaggle, **GPU T4 x2, Internet On**. Датасет MechFaithBench должен быть подключён через Add Input.

Что считается:

1. **Чистый проход**: модель отвечает на вопрос по каждой из 3 200 картинок без закрашивания. Пары, где она права на обеих картинках, идут дальше.
2. **Закрашивание**: для каждой такой пары область с объектом закрашивается чёрным, и отдельно четыре раза закрашивается контрольный кусок той же площади в другом месте.
3. **Метрика ER**: насколько закрашивание объекта снижает уверенность в правильном ответе сильнее, чем закрашивание контрольного куска.

Модель на 2 млрд параметров помещается на одну T4, поэтому работают **два процесса, по одному на карту**, каждый берёт свою половину пар.

Кода в ноутбуке нет: он записывает на диск копию репозитория `mechfaith-robustness` и вызывает его скрипты.

**Первый запуск — проверочный** (`SMOKE = True`): по 3 пары на категорию, несколько минут.
Потом `SMOKE = False`. Результаты сохраняются по ходу и раз в 15 минут упаковываются в zip; новая сессия продолжает из архива.

In [1]:
import os, sys, json, time, glob, zipfile, shutil, subprocess

SMOKE = False                     # True: 3 pairs per category through every step
MODEL = "qwen2vl_2b"             # name of a file in configs/models/
OPERATOR, SUPPORT, K = "black", "pair_union", 4
RUN_ENVELOPE = False             # also run the single-box support (pair_envelope) for comparison
RUN_NATIVE_CHECK = True          # redo counting + relation without downscaling the 1024x1024 images
CHECKPOINT_MINUTES = 15
RESTORE_ZIP = None               # explicit path to an archive of a previous run; None = search /kaggle/input

IS_KAGGLE = os.path.exists("/kaggle/working")
BASE = "/kaggle/working" if IS_KAGGLE else os.path.abspath("kaggle_sim")
REPO = os.path.join(BASE, "mfrob_smoke" if SMOKE else "mfrob")
ZIP_PATH = os.path.join(BASE, f"mfrob_outputs_{MODEL}{'_smoke' if SMOKE else ''}.zip")
os.makedirs(REPO, exist_ok=True)

import torch
N_GPUS = max(torch.cuda.device_count(), 1)
print("repo:", REPO, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none")
if IS_KAGGLE:
    get_ipython().system('pip install -q -U "transformers>=4.49.0" "accelerate>=0.26.0"')

repo: /kaggle/working/mfrob | GPUs: ['Tesla T4', 'Tesla T4']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 36.4 MB/s eta 0:00:00


In [2]:
FILES = {"src/mfrob/__init__.py": "\"\"\"Robustness of the MechFaithBench behavioural protocol.\"\"\"\n", "src/mfrob/analysis/__init__.py": "", "src/mfrob/data/__init__.py": "", "src/mfrob/data/release.py": "\"\"\"Loading the MechFaithBench v1.9 release into Record objects.\"\"\"\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom PIL import Image\n\nSIDES = (\"original\", \"edited\")\nCATEGORY_SHORT = {\n    \"add\": \"add\", \"remove\": \"remove\", \"color_alter\": \"color\", \"counting\": \"counting\", \"relation\": \"relation\",\n    \"replace_appearance\": \"appearance\", \"textual_change\": \"text\", \"surface_material_change\": \"material\",\n}\nCATEGORY_ORDER = [\"add\", \"remove\", \"color\", \"counting\", \"relation\", \"appearance\", \"text\", \"material\"]\n\n\n@dataclass\nclass SideEvidence:\n    absent: bool                     # absence_by_design: the object is not on this side\n    mask_path: Path | None           # evidence mask; None if absent\n    boxes: list[tuple[int, int, int, int]]   # evidence component boxes, half-open xyxy; [] if absent\n    bbox: tuple[int, int, int, int] | None   # envelope of the evidence mask\n    area: int                        # evidence mask area in pixels; 0 if absent\n\n\n@dataclass\nclass Record:\n    record_id: str\n    category: str                    # release name, e.g. \"replace_appearance\"\n    group_id: str                    # sha256 of the original image; records sharing an image share it\n    image_path: dict[str, Path]\n    size: tuple[int, int] | None     # (W, H), identical on both sides; None in meta-only mode\n    question: str\n    gold: dict[str, str]             # side -> \"yes\" | \"no\"\n    evidence: dict[str, SideEvidence]\n    envelope: tuple[int, int, int, int]      # roi_bbox_x1..y2, the cross-side union envelope\n    strict_eligible: bool\n    protocol: str\n    aux_questions: dict[str, str] = field(default_factory=dict)\n\n    @property\n    def cat(self) -> str:\n        return CATEGORY_SHORT[self.category]\n\n\ndef find_release_root(config_path: str | Path = \"configs/data.yaml\") -> Path:\n    \"\"\"First configured release path that exists, else the folder of the first annotations.csv found.\"\"\"\n    cfg = yaml.safe_load(Path(config_path).read_text())\n    for p in cfg.get(\"release_roots\", []):\n        if (Path(p) / \"annotations.csv\").exists():\n            return Path(p)\n    for root in cfg.get(\"search_roots\", []):\n        hits = sorted(Path(root).rglob(\"annotations.csv\")) if Path(root).is_dir() else []\n        if hits:\n            return hits[0].parent\n    raise FileNotFoundError(\"MechFaithBench release not found; see configs/data.yaml and scripts/download_release.py\")\n\n\ndef read_checksums(root: Path) -> dict[str, str]:\n    out = {}\n    for line in (root / \"checksums.sha256\").read_text().splitlines():\n        if line.strip():\n            digest, path = line.split(None, 1)\n            out[path.lstrip(\"*\").strip()] = digest\n    return out\n\n\ndef _box(b) -> tuple[int, int, int, int] | None:\n    return None if b is None else tuple(int(v) for v in b)\n\n\ndef _side_evidence(root: Path, side: dict) -> SideEvidence:\n    # Nine absent sides in `counting` have no evidence_* keys at all, hence .get everywhere.\n    mask = side.get(\"evidence_mask\")\n    return SideEvidence(\n        absent=bool(side.get(\"absence_by_design\", False)),\n        mask_path=(root / mask) if mask else None,\n        boxes=[_box(b) for b in (side.get(\"evidence_component_bboxes_xyxy\") or [])],\n        bbox=_box(side.get(\"evidence_bbox_xyxy\")),\n        area=int(side.get(\"evidence_area_pixels\") or 0),\n    )\n\n\ndef image_size(path: Path) -> tuple[int, int]:\n    with Image.open(path) as im:\n        return im.size\n\n\ndef load_release(root: str | Path, with_sizes: bool = True) -> list[Record]:\n    \"\"\"All 1,600 records. with_sizes=False reads metadata only (no image files needed).\"\"\"\n    root = Path(root)\n    ann = pd.read_csv(root / \"annotations.csv\", keep_default_na=False)\n    roi = {r[\"record_id\"]: r for r in json.loads((root / \"roi_annotations.json\").read_text())[\"records\"]}\n    checksums = read_checksums(root)\n    records = []\n    for row in ann.itertuples():\n        r = roi[row.record_id]\n        paths = {\"original\": root / row.original_image, \"edited\": root / row.edited_image}\n        size = None\n        if with_sizes:\n            size = image_size(paths[\"original\"])\n            other = image_size(paths[\"edited\"])\n            if other != size:\n                raise ValueError(f\"{row.record_id}: sides differ in size, {size} vs {other}\")\n        aux = {}\n        for d in json.loads(row.diagnostic_questions_json or \"[]\"):\n            if d.get(\"type\") == \"non_flip_control\":\n                aux[\"non_flip\"] = d[\"question\"]\n        for k in (1, 2, 3):\n            aux[f\"probe{k}\"] = getattr(row, f\"hallucination_probe_{k}\")\n        records.append(Record(\n            record_id=row.record_id,\n            category=row.category,                         # never derive it from the id prefix or the folder\n            group_id=checksums[row.original_image],\n            image_path=paths,\n            size=size,\n            question=row.question,\n            gold={\"original\": row.gold_original, \"edited\": row.gold_edited},\n            evidence={s: _side_evidence(root, r[s]) for s in SIDES},\n            envelope=(int(row.roi_bbox_x1), int(row.roi_bbox_y1), int(row.roi_bbox_x2), int(row.roi_bbox_y2)),\n            strict_eligible=row.roi_strict_spatial_eligible == \"yes\",\n            protocol=row.roi_evidence_protocol,\n            aux_questions=aux,\n        ))\n    return records\n\n\n@lru_cache(maxsize=4096)\ndef load_mask(path: Path) -> np.ndarray:\n    \"\"\"Evidence mask as a bool array (H, W); cached.\"\"\"\n    return np.asarray(Image.open(path).convert(\"L\")) > 127\n", "src/mfrob/geometry/__init__.py": "", "src/mfrob/geometry/controls.py": "\"\"\"Control regions: same pixel count as the support, away from the evidence.\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\nfrom scipy import ndimage\n\nFOUR = ndimage.generate_binary_structure(2, 1)     # 4-connectivity\n\n\ndef seed_from(*parts) -> int:\n    \"\"\"Reproducible seed; Python's hash() is salted per process and must not be used.\"\"\"\n    return int.from_bytes(hashlib.sha256(\"|\".join(map(str, parts)).encode()).digest()[:8], \"big\")\n\n\ndef big_enough_components(forbidden: np.ndarray, n: int) -> np.ndarray:\n    \"\"\"Pixels of the free 4-connected components that hold at least n pixels.\"\"\"\n    lab, k = ndimage.label(~forbidden, structure=FOUR)\n    if k == 0:\n        return np.zeros_like(forbidden)\n    sizes = np.bincount(lab.ravel())\n    sizes[0] = 0\n    return np.isin(lab, np.nonzero(sizes >= n)[0])\n\n\ndef growth_feasible(forbidden: np.ndarray, n: int) -> bool:\n    return bool(big_enough_components(forbidden, n).any())\n\n\ndef growth(forbidden: np.ndarray, n: int, seed: int) -> np.ndarray | None:\n    \"\"\"Exactly n pixels grown breadth-first (4-connectivity) from a random free seed; None if infeasible.\n    Whole layers are added in order; the last layer is filled with a random subset.\"\"\"\n    ok = big_enough_components(forbidden, n)\n    if not ok.any():\n        return None\n    rng = np.random.default_rng(seed)\n    h, w = forbidden.shape\n    ys, xs = np.nonzero(ok)\n    j = int(rng.integers(len(ys)))\n    cur = np.zeros((h, w), bool)\n    cur[ys[j], xs[j]] = True\n    count, y0, y1, x0, x1 = 1, ys[j], ys[j] + 1, xs[j], xs[j] + 1\n    while count < n:\n        # work on the bounding window of the current blob, grown by one pixel\n        y0, y1, x0, x1 = max(0, y0 - 1), min(h, y1 + 1), max(0, x0 - 1), min(w, x1 + 1)\n        c, free = cur[y0:y1, x0:x1], ok[y0:y1, x0:x1]\n        nb = np.zeros_like(c)\n        nb[1:] |= c[:-1]; nb[:-1] |= c[1:]; nb[:, 1:] |= c[:, :-1]; nb[:, :-1] |= c[:, 1:]\n        frontier = nb & free & ~c\n        k = int(frontier.sum())\n        if k == 0:                                   # cannot happen inside a component with >= n pixels\n            return None\n        if count + k <= n:\n            c |= frontier\n            count += k\n        else:\n            fy, fx = np.nonzero(frontier)\n            pick = rng.choice(k, n - count, replace=False)\n            c[fy[pick], fx[pick]] = True\n            count = n\n    return cur\n", "src/mfrob/geometry/supports.py": "\"\"\"Occlusion regions (supports) as bool masks of shape (H, W).\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\nfrom scipy import ndimage\n\nfrom ..data.release import SIDES, Record\n\nMARGIN = 8          # px between a control and any evidence; see PLAN.md, section 3.4\n\n\ndef rasterize(boxes, size: tuple[int, int]) -> np.ndarray:\n    \"\"\"Union of half-open xyxy boxes; size is (W, H).\"\"\"\n    w, h = size\n    m = np.zeros((h, w), bool)\n    for x1, y1, x2, y2 in boxes:\n        m[max(0, y1):min(h, y2), max(0, x1):min(w, x2)] = True\n    return m\n\n\ndef pair_union(rec: Record) -> np.ndarray:\n    \"\"\"Union of the evidence component boxes of both sides; the same region on both sides (reference support).\"\"\"\n    return rasterize([b for s in SIDES for b in rec.evidence[s].boxes], rec.size)\n\n\ndef pair_envelope(rec: Record) -> np.ndarray:\n    \"\"\"The single cross-side envelope box roi_bbox_x1..y2.\"\"\"\n    return rasterize([rec.envelope], rec.size)\n\n\nSUPPORTS = {\"pair_union\": pair_union, \"pair_envelope\": pair_envelope}\n\n\ndef dilate(mask: np.ndarray, r: int) -> np.ndarray:\n    \"\"\"Dilation by a (2r+1) x (2r+1) square, clipped to the image.\"\"\"\n    if r <= 0:\n        return mask\n    return ndimage.maximum_filter(mask.astype(np.uint8), size=2 * r + 1, mode=\"constant\", cval=0).astype(bool)\n\n\ndef forbidden_zone(support: np.ndarray, union: np.ndarray, margin: int = MARGIN) -> np.ndarray:\n    \"\"\"Where a control may not go: the support and all annotated evidence of both sides, plus a margin.\"\"\"\n    return dilate(support | union, margin)\n\n\ndef mask_hash(mask: np.ndarray) -> str:\n    return hashlib.sha1(np.packbits(mask).tobytes() + str(mask.shape).encode()).hexdigest()\n\n\ndef eccentricity(mask: np.ndarray) -> float:\n    \"\"\"Distance from the centre of the mask's bounding rectangle to the image centre, in half-diagonals.\"\"\"\n    ys, xs = np.nonzero(mask)\n    h, w = mask.shape\n    cy, cx = (h - 1) / 2, (w - 1) / 2\n    return float(np.hypot((ys.min() + ys.max()) / 2 - cy, (xs.min() + xs.max()) / 2 - cx) / np.hypot(cy, cx))\n", "src/mfrob/metrics/__init__.py": "", "src/mfrob/metrics/er.py": "\"\"\"Evidence Reliance and related quantities from saved logits (probability scale).\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom ..data.release import SIDES\n\n\ndef sigmoid(x):\n    return 1.0 / (1.0 + np.exp(-np.asarray(x, float)))\n\n\ndef pair_er(interv: pd.DataFrame, gold: dict[tuple[str, str], str]) -> pd.DataFrame:\n    \"\"\"One row per pair with target + controls: ER, its YES/NO parts and ROA. Negative values are kept.\n\n    gold maps (record_id, side) -> \"yes\" | \"no\". p(gold) = sigmoid(+ld) on a yes side, sigmoid(-ld) on a no side.\n    ER_side = mean_k p(gold | control_k) - p(gold | target);  ER = (ER_yes + ER_no) / 2.\n    ROA = 1 if the summed gold probability under the target is below that under every control.\n    \"\"\"\n    d = interv.drop_duplicates([\"record_id\", \"side\", \"region_kind\", \"k\"], keep=\"last\").copy()\n    sign = np.array([1.0 if gold[(r, s)] == \"yes\" else -1.0 for r, s in zip(d.record_id, d.side)])\n    d[\"p_gold\"] = sigmoid(sign * d.ld.values)\n    d[\"gold\"] = np.where(sign > 0, \"yes\", \"no\")\n    rows = []\n    for rid, g in d.groupby(\"record_id\", sort=False):\n        t = g[g.region_kind == \"target\"].set_index(\"side\").p_gold\n        c = g[g.region_kind == \"control\"]\n        if len(t) != 2 or c.empty:\n            continue\n        cm = c.groupby(\"side\").p_gold.mean()\n        side_gold = g.drop_duplicates(\"side\").set_index(\"side\").gold\n        er = {side_gold[s]: float(cm[s] - t[s]) for s in SIDES}\n        per_k = c.pivot(index=\"k\", columns=\"side\", values=\"p_gold\").sum(1)\n        rows.append({\"record_id\": rid, \"er\": (er[\"yes\"] + er[\"no\"]) / 2, \"er_yes\": er[\"yes\"], \"er_no\": er[\"no\"],\n                     \"roa\": bool(float(t.sum()) < float(per_k.min())), \"k\": int(per_k.shape[0]),\n                     \"p_target_yes\": float(t[side_gold[side_gold == \"yes\"].index[0]]),\n                     \"p_control_yes\": float(cm[side_gold[side_gold == \"yes\"].index[0]]),\n                     \"n_pixels\": int(g.n_pixels.iloc[0])})\n    return pd.DataFrame(rows)\n\n\ndef summarize(er: pd.DataFrame, pairs: pd.DataFrame, n_eligible: int | None = None) -> dict:\n    \"\"\"Micro means over pairs. pairs: clean-pass table with B per record. LVF = coverage x ER over the eligible set.\"\"\"\n    out = {\"pairs\": int(len(er)), \"er\": float(er.er.mean()), \"er_yes\": float(er.er_yes.mean()),\n           \"er_no\": float(er.er_no.mean()), \"roa\": float(er.roa.mean())}\n    if n_eligible:\n        out[\"coverage\"] = len(er) / n_eligible\n        out[\"lvf\"] = out[\"coverage\"] * out[\"er\"]\n    return out\n", "src/mfrob/models/__init__.py": "", "src/mfrob/models/base.py": "\"\"\"Common interface of the scored vision-language models.\"\"\"\nfrom __future__ import annotations\n\nfrom abc import ABC, abstractmethod\n\nimport torch\nfrom PIL import Image\n\nYES_SPELLINGS = [\"Yes\", \"yes\", \" Yes\", \" yes\"]\nNO_SPELLINGS = [\"No\", \"no\", \" No\", \" no\"]\n\n\ndef pick_device() -> str:\n    if torch.cuda.is_available():\n        return \"cuda\"\n    if torch.backends.mps.is_available():\n        return \"mps\"\n    return \"cpu\"\n\n\ndef pick_dtype(device: str, requested: str = \"auto\") -> torch.dtype:\n    \"\"\"auto: bf16 on Ampere+ GPUs, fp16 on older CUDA GPUs, fp32 on mps and cpu.\"\"\"\n    if requested != \"auto\":\n        return getattr(torch, requested)\n    if device == \"cuda\":\n        return torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\n    return torch.float32\n\n\ndef single_token_ids(tokenizer, spellings: list[str]) -> list[int]:\n    \"\"\"Ids of the spellings that the tokenizer encodes as exactly one token.\"\"\"\n    ids = set()\n    for w in spellings:\n        enc = tokenizer.encode(w, add_special_tokens=False)\n        if len(enc) == 1:\n            ids.add(enc[0])\n    return sorted(ids)\n\n\ndef score_from_logits(logits: torch.Tensor, yes_ids: list[int], no_ids: list[int]) -> dict:\n    \"\"\"Yes/no summary of next-token logits (1-D, any dtype). Everything is computed in fp32.\"\"\"\n    l = logits.float()\n    if not torch.isfinite(l).all():\n        raise FloatingPointError(\"non-finite logits\")\n    l_yes, l_no = torch.logsumexp(l[yes_ids], 0), torch.logsumexp(l[no_ids], 0)\n    top1 = int(l.argmax())\n    return {\n        \"ld\": float(l_yes - l_no), \"l_yes\": float(l_yes), \"l_no\": float(l_no),\n        \"logits_yes\": [float(v) for v in l[yes_ids]], \"logits_no\": [float(v) for v in l[no_ids]],\n        \"lse_all\": float(torch.logsumexp(l, 0)), \"top1_id\": top1, \"top1_is_yesno\": top1 in yes_ids or top1 in no_ids,\n    }\n\n\nclass VLM(ABC):\n    name: str\n    hf_id: str\n    revision: str | None\n    device: str\n    dtype: torch.dtype\n    max_pixels: int\n    prompt_template: str\n\n    @abstractmethod\n    def yes_no_ids(self) -> tuple[list[int], list[int]]: ...\n\n    @abstractmethod\n    def score(self, images: list[Image.Image], question: str) -> list[dict]:\n        \"\"\"One forward pass without generation; one dict per image (see score_from_logits) plus n_img_tokens.\"\"\"\n", "src/mfrob/models/qwen_vl.py": "\"\"\"Qwen2-VL / Qwen2.5-VL scoring.\"\"\"\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport torch\nfrom PIL import Image\nfrom transformers import AutoModelForImageTextToText, AutoProcessor\n\nfrom .base import NO_SPELLINGS, VLM, YES_SPELLINGS, pick_device, pick_dtype, score_from_logits, single_token_ids\n\n\nclass QwenVL(VLM):\n    def __init__(self, cfg: dict):\n        self.name, self.hf_id, self.max_pixels = cfg[\"name\"], cfg[\"hf_id\"], int(cfg[\"max_pixels\"])\n        self.device = cfg.get(\"device\") or pick_device()\n        self.dtype = pick_dtype(self.device, cfg.get(\"dtype\", \"auto\"))\n        kw = {\"device_map\": \"auto\"} if self.device == \"cuda\" else {}\n        self.model = AutoModelForImageTextToText.from_pretrained(self.hf_id, dtype=self.dtype, **kw).eval()\n        if self.device != \"cuda\":\n            self.model.to(self.device)\n        self.processor = AutoProcessor.from_pretrained(self.hf_id, max_pixels=self.max_pixels)\n        self.revision = getattr(self.model.config, \"_commit_hash\", None)\n        if self.revision is None:                      # newer transformers: take it from the cache path\n            from huggingface_hub import snapshot_download\n            self.revision = Path(snapshot_download(self.hf_id, local_files_only=True)).name\n        tok = self.processor.tokenizer\n        self._yes, self._no = single_token_ids(tok, YES_SPELLINGS), single_token_ids(tok, NO_SPELLINGS)\n        assert self._yes and self._no\n        self.image_token_id = getattr(self.model.config, \"image_token_id\", None) or tok.convert_tokens_to_ids(\"<|image_pad|>\")\n        self.prompt_template = self._text(\"{question}\")\n        self._in_device = next(self.model.parameters()).device\n\n    def _text(self, question: str) -> str:\n        messages = [{\"role\": \"user\", \"content\": [{\"type\": \"image\"}, {\"type\": \"text\", \"text\": question}]}]\n        return self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n    def yes_no_ids(self) -> tuple[list[int], list[int]]:\n        return self._yes, self._no\n\n    @torch.no_grad()\n    def score(self, images: list[Image.Image], question: str) -> list[dict]:\n        # Images of one batch must have the same size: then every sequence has the same length and\n        # no padding is involved, so the last position is the answer position for all of them.\n        assert len({im.size for im in images}) == 1, \"images of one batch must share a size\"\n        inputs = self.processor(text=[self._text(question)] * len(images), images=[im.convert(\"RGB\") for im in images],\n                                return_tensors=\"pt\", padding=True).to(self._in_device)\n        assert bool(inputs[\"attention_mask\"].all()), \"unexpected padding\"\n        logits = self.model(**inputs, use_cache=False).logits[:, -1]\n        n_img = (inputs[\"input_ids\"] == self.image_token_id).sum(1).tolist()\n        out = []\n        for i in range(len(images)):\n            d = score_from_logits(logits[i].cpu(), self._yes, self._no)\n            d[\"n_img_tokens\"] = int(n_img[i])\n            out.append(d)\n        return out\n", "src/mfrob/operators/__init__.py": "", "src/mfrob/operators/pixel.py": "\"\"\"Pixel-level occlusion operators. Pixels outside the region are never touched.\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nfrom PIL import Image\n\nOPERATORS = (\"black\", \"gray\", \"mean\", \"noise\")\n\n\ndef apply(image: Image.Image, region: np.ndarray, op: str, seed: int = 0) -> Image.Image:\n    a = np.asarray(image.convert(\"RGB\")).copy()\n    assert region.shape == a.shape[:2], (region.shape, a.shape)\n    if op == \"black\":\n        a[region] = 0\n    elif op == \"gray\":\n        a[region] = 128\n    elif op == \"mean\":\n        a[region] = np.rint(a.reshape(-1, 3).mean(0)).astype(np.uint8)\n    elif op == \"noise\":\n        a[region] = np.random.default_rng(seed).integers(0, 256, (int(region.sum()), 3), dtype=np.uint8)\n    else:\n        raise ValueError(f\"unknown operator {op}\")\n    return Image.fromarray(a)\n", "src/mfrob/run/__init__.py": "", "src/mfrob/run/clean_pass.py": "\"\"\"Clean pass: both sides of every pair, primary question, no occlusion. Resumable.\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nfrom ..data.release import SIDES, Record\nfrom ..models.base import VLM\n\n\ndef image_key(record_id: str, side: str, operator: str = \"clean\", region_hash: str = \"\") -> str:\n    return hashlib.sha1(f\"{record_id}|{side}|{operator}|{region_hash}\".encode()).hexdigest()\n\n\ndef load_logits(out_dir: Path) -> pd.DataFrame:\n    parts = sorted(Path(out_dir).glob(\"part-*.parquet\"))\n    return pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True) if parts else pd.DataFrame()\n\n\ndef run_clean_pass(model: VLM, records: list[Record], out_dir: Path, shard_size: int = 200, log_every: int = 50, tag: str = \"\") -> None:\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    done = load_logits(out_dir)\n    done_ids = set(done[done.question_kind == \"primary\"].record_id) if len(done) else set()\n    todo = [r for r in records if r.record_id not in done_ids]\n    print(f\"clean pass: {len(records)} pairs, {len(done_ids)} already done, {len(todo)} to go\")\n    rows, t0 = [], time.time()\n    n_parts = len(list(out_dir.glob(f\"part-{tag}*.parquet\")))\n\n    def flush():\n        nonlocal rows, n_parts\n        if rows:\n            pd.DataFrame(rows).to_parquet(out_dir / f\"part-{tag}{n_parts:05d}.parquet\", index=False)\n            n_parts += 1\n            rows = []\n\n    for i, rec in enumerate(todo, 1):\n        images = [Image.open(rec.image_path[s]) for s in SIDES]\n        scores = model.score(images, rec.question)\n        assert scores[0][\"n_img_tokens\"] == scores[1][\"n_img_tokens\"]\n        for side, sc in zip(SIDES, scores):\n            rows.append({\"model\": model.name, \"record_id\": rec.record_id, \"side\": side, \"question_kind\": \"primary\",\n                         \"image_key\": image_key(rec.record_id, side), **sc, \"dtype\": str(model.dtype),\n                         \"max_pixels\": model.max_pixels, \"revision\": model.revision,\n                         \"timestamp\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\")})\n        if i % shard_size == 0:\n            flush()\n        if i % log_every == 0 or i == len(todo):\n            el = time.time() - t0\n            print(f\"  {i}/{len(todo)} pairs | {el / (2 * i):.2f} s per forward | left ~{el / i * (len(todo) - i) / 60:.0f} min\", flush=True)\n    flush()\n", "src/mfrob/run/intervene.py": "\"\"\"Interventions: occlude the evidence region (target) and K matched control regions, score every variant.\"\"\"\nfrom __future__ import annotations\n\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nfrom ..data.release import SIDES, Record\nfrom ..geometry.controls import growth, seed_from\nfrom ..geometry.supports import SUPPORTS, eccentricity, forbidden_zone, mask_hash, pair_union\nfrom ..models.base import VLM\nfrom ..operators.pixel import apply\nfrom .clean_pass import image_key, load_logits\n\n\ndef build_regions(rec: Record, support: str = \"pair_union\", k: int = 4, margin: int = 8):\n    \"\"\"(target mask, [control masks]) for a pair-fixed support, or (target, None) if no control fits.\"\"\"\n    target = SUPPORTS[support](rec)\n    forb = forbidden_zone(target, pair_union(rec), margin)\n    n = int(target.sum())\n    controls = []\n    for i in range(k):\n        c = growth(forb, n, seed_from(rec.record_id, \"pair\", support, \"growth\", i))\n        if c is None:\n            return target, None\n        controls.append(c)\n    return target, controls\n\n\ndef run_interventions(model: VLM, records: list[Record], out_dir: Path, operator: str = \"black\", support: str = \"pair_union\",\n                      k: int = 4, margin: int = 8, tag: str = \"\", shard_size: int = 100, log_every: int = 25) -> None:\n    \"\"\"records: pairs with B = 1. Pairs without a feasible control are written to infeasible-*.txt and skipped.\"\"\"\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    done = load_logits(out_dir)\n    done_ids = set(done.record_id) if len(done) else set()\n    skipped = {l.strip() for f in out_dir.glob(\"infeasible-*.txt\") for l in f.read_text().splitlines() if l.strip()}\n    todo = [r for r in records if r.record_id not in done_ids and r.record_id not in skipped]\n    print(f\"interventions [{operator}, {support}, growth, K={k}]: {len(records)} pairs, {len(done_ids)} done, {len(todo)} to go\")\n    rows, t0, n_fwd = [], time.time(), 0\n    n_parts = len(list(out_dir.glob(f\"part-{tag}*.parquet\")))\n\n    def flush():\n        nonlocal rows, n_parts\n        if rows:\n            pd.DataFrame(rows).to_parquet(out_dir / f\"part-{tag}{n_parts:05d}.parquet\", index=False)\n            n_parts += 1\n            rows = []\n\n    for i, rec in enumerate(todo, 1):\n        target, controls = build_regions(rec, support, k, margin)\n        if controls is None:\n            with open(out_dir / f\"infeasible-{tag or 'all'}.txt\", \"a\") as f:\n                f.write(rec.record_id + \"\\n\")\n            continue\n        regions = [(\"target\", -1, target)] + [(\"control\", j, c) for j, c in enumerate(controls)]\n        n_pix = int(target.sum())\n        assert all(int(m.sum()) == n_pix for _, _, m in regions)\n        for side in SIDES:\n            clean = Image.open(rec.image_path[side])\n            images = [apply(clean, m, operator, seed_from(rec.record_id, side, operator, mask_hash(m))) for _, _, m in regions]\n            scores = model.score(images, rec.question)\n            n_fwd += len(images)\n            for (kind, j, m), sc in zip(regions, scores):\n                rows.append({\"model\": model.name, \"record_id\": rec.record_id, \"side\": side, \"question_kind\": \"primary\",\n                             \"operator\": operator, \"support\": support, \"region_kind\": kind, \"control_type\": \"growth\" if kind == \"control\" else None,\n                             \"k\": j, \"n_pixels\": n_pix, \"eccentricity\": eccentricity(m),\n                             \"image_key\": image_key(rec.record_id, side, operator, mask_hash(m)), **sc,\n                             \"dtype\": str(model.dtype), \"max_pixels\": model.max_pixels, \"revision\": model.revision,\n                             \"timestamp\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\")})\n        if i % shard_size == 0:\n            flush()\n        if i % log_every == 0 or i == len(todo):\n            el = time.time() - t0\n            print(f\"  {i}/{len(todo)} pairs | {el / max(n_fwd, 1):.2f} s per forward | left ~{el / i * (len(todo) - i) / 60:.0f} min\", flush=True)\n    flush()\n", "scripts/analyze_clean.py": "#!/usr/bin/env python3\n\"\"\"Summary of a clean pass: pair accuracy, ordering inside the pair, error types, by category.\n\n    python scripts/analyze_clean.py --model qwen2vl_2b\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import CATEGORY_ORDER, SIDES, find_release_root, load_release  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\n\n\ndef pair_table(logits: pd.DataFrame, records) -> pd.DataFrame:\n    \"\"\"One row per pair: B (correct on both clean sides), ordering, error type.\"\"\"\n    d = logits[logits.question_kind == \"primary\"].drop_duplicates([\"record_id\", \"side\"], keep=\"last\")\n    w = d.pivot(index=\"record_id\", columns=\"side\", values=[\"ld\", \"top1_is_yesno\", \"n_img_tokens\"])\n    rows = []\n    for r in records:\n        if r.record_id not in w.index:\n            continue\n        ld = {s: float(w.loc[r.record_id, (\"ld\", s)]) for s in SIDES}\n        valid = all(bool(w.loc[r.record_id, (\"top1_is_yesno\", s)]) for s in SIDES)\n        yes_side = \"original\" if r.gold[\"original\"] == \"yes\" else \"edited\"\n        no_side = \"edited\" if yes_side == \"original\" else \"original\"\n        ld_yes, ld_no = ld[yes_side], ld[no_side]\n        rows.append({\"record_id\": r.record_id, \"cat\": r.cat, \"group_id\": r.group_id, \"ld_yes\": ld_yes, \"ld_no\": ld_no,\n                     \"B\": valid and ld_yes > 0 and ld_no <= 0, \"ordering_ok\": ld_yes > ld_no,\n                     \"yes_yes\": ld_yes > 0 and ld_no > 0, \"no_no\": ld_yes <= 0 and ld_no <= 0,\n                     \"valid_tokens\": valid, \"n_img_tokens\": int(w.loc[r.record_id, (\"n_img_tokens\", \"original\")]),\n                     \"strict_eligible\": r.strict_eligible})\n    return pd.DataFrame(rows)\n\n\ndef boot_ci(x, groups, n_boot=2000, seed=20260803):\n    \"\"\"Mean with a 95% cluster bootstrap interval (clusters = image groups).\"\"\"\n    x = np.asarray(x, float); codes, _ = pd.factorize(groups)\n    sums = np.bincount(codes, weights=x); cnt = np.bincount(codes)\n    rng = np.random.default_rng(seed); idx = rng.integers(0, len(cnt), (n_boot, len(cnt)))\n    m = sums[idx].sum(1) / cnt[idx].sum(1)\n    return float(x.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    args = ap.parse_args()\n    run = Path(\"outputs/logits\") / args.model / \"clean\"\n    records = load_release(find_release_root(), with_sizes=False)\n    P = pair_table(load_logits(run), records)\n    cols = [\"B\", \"ordering_ok\", \"yes_yes\", \"no_no\"]\n    T = P.groupby(\"cat\")[cols].mean().reindex(CATEGORY_ORDER)\n    T.insert(0, \"pairs\", P.groupby(\"cat\").size().reindex(CATEGORY_ORDER))\n    T.loc[\"ALL\"] = [len(P)] + [P[c].mean() for c in cols]\n    T = T.rename(columns={\"B\": \"pair accuracy (PA)\", \"ordering_ok\": \"right ordering\", \"yes_yes\": \"yes on both\", \"no_no\": \"no on both\"})\n    pd.set_option(\"display.width\", 200)\n    print(T.round(3).to_string())\n    pa, order = boot_ci(P.B, P.group_id), boot_ci(P.ordering_ok, P.group_id)\n    print(f\"\\nPA {pa[0]:.3f} [{pa[1]:.3f}, {pa[2]:.3f}] | right ordering {order[0]:.3f} [{order[1]:.3f}, {order[2]:.3f}]\"\n          f\" | pairs with B = 1: {int(P.B.sum())} of {len(P)} | top-1 token is yes/no: {P.valid_tokens.mean():.4f}\")\n    out = Path(\"outputs/reports\"); out.mkdir(parents=True, exist_ok=True)\n    P.to_parquet(run / \"pairs.parquet\", index=False)\n    meta = json.loads((run / \"run_meta.json\").read_text())\n    (out / f\"clean_{args.model}.json\").write_text(json.dumps({\"run\": meta, \"pa\": pa, \"right_ordering\": order,\n        \"by_category\": T.round(4).reset_index().to_dict(orient=\"records\")}, ensure_ascii=False, indent=1))\n    print(\"saved\", run / \"pairs.parquet\", \"and\", out / f\"clean_{args.model}.json\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/analyze_er.py": "#!/usr/bin/env python3\n\"\"\"ER, ER_YES, ER_NO, ROA, LVF of one experiment.   python scripts/analyze_er.py --model qwen2vl_2b\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom analyze_clean import boot_ci, pair_table  # noqa: E402\nfrom mfrob.data.release import CATEGORY_ORDER, SIDES, find_release_root, load_release  # noqa: E402\nfrom mfrob.metrics.er import pair_er  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--experiment\", default=\"black__pair_union__growth_k4\")\n    args = ap.parse_args()\n    base = Path(\"outputs/logits\") / args.model\n    records = load_release(find_release_root(), with_sizes=False)\n    gold = {(r.record_id, s): r.gold[s] for r in records for s in SIDES}\n    P = pair_table(load_logits(base / \"clean\"), records).set_index(\"record_id\")\n    E = pair_er(load_logits(base / args.experiment), gold).set_index(\"record_id\")\n    E[\"cat\"], E[\"group_id\"] = P.cat.reindex(E.index), P.group_id.reindex(E.index)\n    infeasible = {l.strip() for f in (base / args.experiment).glob(\"infeasible-*.txt\") for l in f.read_text().splitlines() if l.strip()}\n    n_eligible = len(P) - len({r for r in P.index if r in infeasible}) - 0      # among clean-correct pairs only the tried ones are known\n    T = E.groupby(\"cat\")[[\"er\", \"er_yes\", \"er_no\", \"roa\"]].mean().reindex(CATEGORY_ORDER)\n    T.insert(0, \"pairs\", E.groupby(\"cat\").size().reindex(CATEGORY_ORDER))\n    T.loc[\"ALL\"] = [len(E)] + [E[c].mean() for c in (\"er\", \"er_yes\", \"er_no\", \"roa\")]\n    pd.set_option(\"display.width\", 200)\n    print(T.round(3).to_string())\n    ci = {c: boot_ci(E[c], E.group_id) for c in (\"er\", \"er_yes\", \"er_no\", \"roa\")}\n    print(\"\\n\" + \" | \".join(f\"{c} {v[0]:.3f} [{v[1]:.3f}, {v[2]:.3f}]\" for c, v in ci.items()))\n    print(f\"pairs with B = 1: {int(P.B.sum())} | with a control: {len(E)} | without: {len(infeasible)} | K = {int(E.k.max())}\")\n    out = Path(\"outputs/reports\"); out.mkdir(parents=True, exist_ok=True)\n    E.reset_index().to_parquet(base / args.experiment / \"pair_er.parquet\", index=False)\n    (out / f\"er_{args.model}_{args.experiment}.json\").write_text(json.dumps(\n        {\"ci\": ci, \"by_category\": T.round(4).reset_index().to_dict(orient=\"records\"), \"infeasible\": len(infeasible)}, indent=1))\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/audit.py": "#!/usr/bin/env python3\n\"\"\"Checks the release against the facts listed in PLAN.md (stage 1). Exits with 1 on any mismatch.\n\n    python scripts/audit.py                # everything that the files on disk allow\n    python scripts/audit.py --meta-only    # annotations and ROI metadata only\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import SIDES, find_release_root, load_release, read_checksums  # noqa: E402\n\nEXPECT_CATEGORIES = {\"add\": 212, \"remove\": 223, \"replace_appearance\": 256, \"color_alter\": 209, \"counting\": 150,\n                     \"relation\": 110, \"textual_change\": 222, \"surface_material_change\": 218}\nEXPECT_ABSENT = {(\"add\", \"original\"): 212, (\"remove\", \"edited\"): 223, (\"counting\", \"original\"): 1,\n                 (\"counting\", \"edited\"): 8, (\"textual_change\", \"original\"): 3, (\"textual_change\", \"edited\"): 1}\nEXPECT_SIZES = {(512, 512): 454, (1024, 1024): 263, (640, 480): 197}\n\nFAILED = []\n\n\ndef check(name: str, got, want=None) -> None:\n    ok = bool(got) if want is None else got == want\n    print(f\"[{'ok' if ok else 'FAIL'}] {name}: {got}\" + (\"\" if ok or want is None else f\"  (expected {want})\"))\n    if not ok:\n        FAILED.append(name)\n\n\ndef sha256(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--root\", type=Path, default=None)\n    ap.add_argument(\"--meta-only\", action=\"store_true\")\n    args = ap.parse_args()\n    root = args.root or find_release_root()\n    print(\"release:\", root)\n    have_images = not args.meta_only and (root / \"images\").is_dir()\n    have_masks = not args.meta_only and (root / \"segmentation\").is_dir()\n    recs = load_release(root, with_sizes=have_images)\n\n    # --- metadata\n    check(\"unique record ids\", len({r.record_id for r in recs}), 1600)\n    check(\"categories\", dict(Counter(r.category for r in recs)), EXPECT_CATEGORIES)\n    gold = Counter((r.gold[\"original\"], r.gold[\"edited\"]) for r in recs)\n    check(\"gold (original, edited)\", dict(gold), {(\"no\", \"yes\"): 1376, (\"yes\", \"no\"): 224})\n    yes_orig = Counter(r.category for r in recs if r.gold[\"original\"] == \"yes\")\n    check(\"categories with yes on the original side\", dict(yes_orig), {\"remove\": 223, \"textual_change\": 1})\n    check(\"strict eligible\", sum(r.strict_eligible for r in recs), 1591)\n    check(\"protocols\", dict(Counter(r.protocol for r in recs)),\n          {\"target_only\": 1538, \"target_plus_reference\": 53, \"attribute_specific\": 9})\n    check(\"target_plus_reference only in relation\",\n          {r.category for r in recs if r.protocol == \"target_plus_reference\"}, {\"relation\"})\n    absent = Counter((r.category, s) for r in recs for s in SIDES if r.evidence[s].absent)\n    check(\"absent sides\", dict(absent), EXPECT_ABSENT)\n    check(\"no record absent on both sides\", not any(all(r.evidence[s].absent for s in SIDES) for r in recs))\n    check(\"absent sides carry no evidence\",\n          all(e.mask_path is None and not e.boxes and e.area == 0 for r in recs for e in r.evidence.values() if e.absent))\n    check(\"present sides carry a mask, boxes and an area\",\n          all(e.mask_path is not None and e.boxes and e.bbox and e.area > 0\n              for r in recs for e in r.evidence.values() if not e.absent))\n    roi = {x[\"record_id\"]: x for x in json.loads((root / \"roi_annotations.json\").read_text())[\"records\"]}\n    check(\"csv envelope equals legacy_union_bbox_xyxy\",\n          all(list(r.envelope) == roi[r.record_id][\"legacy_union_bbox_xyxy\"] for r in recs))\n    check(\"distinct image groups\", len({r.group_id for r in recs}), 1584)\n    check(\"all questions end with 'Answer yes or no.'\", all(r.question.endswith(\"Answer yes or no.\") for r in recs))\n    check(\"every record has a non-flip question\", all(\"non_flip\" in r.aux_questions for r in recs))\n    manifest = json.loads((root / \"manifest.json\").read_text())\n    for name in (\"annotations.csv\", \"roi_annotations.json\"):\n        check(f\"sha256 of {name} matches the manifest\", sha256(root / name) == manifest[\"sha256\"][name])\n\n    # --- images\n    if have_images:\n        checksums = read_checksums(root)\n        bad = [str(p) for r in recs for p in r.image_path.values() if sha256(p) != checksums[str(p.relative_to(root))]]\n        check(\"sha256 of all 3,200 images\", len(bad), 0)\n        sizes = Counter(r.size for r in recs)\n        check(\"distinct image sizes\", len(sizes), 186)\n        check(\"most frequent sizes\", {k: sizes[k] for k in EXPECT_SIZES}, EXPECT_SIZES)\n        big = Counter(r.category for r in recs if r.size == (1024, 1024))\n        check(\"1024x1024 pairs by category\", dict(big), {\"counting\": 150, \"relation\": 110, \"remove\": 2, \"replace_appearance\": 1})\n\n        def inside(b, size):\n            return 0 <= b[0] < b[2] <= size[0] and 0 <= b[1] < b[3] <= size[1]\n        check(\"every box lies inside its image\",\n              all(inside(r.envelope, r.size) and all(inside(b, r.size) for e in r.evidence.values() for b in e.boxes) for r in recs))\n    else:\n        print(\"[skip] image checks: images are not on disk\")\n\n    # --- masks\n    if have_masks:\n        n, bad = 0, []\n        for r in recs:\n            for s in SIDES:\n                e = r.evidence[s]\n                if e.absent:\n                    continue\n                m = np.asarray(Image.open(e.mask_path).convert(\"L\")); n += 1\n                on = m > 127\n                ys, xs = np.nonzero(on)\n                env = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)\n                if not (set(np.unique(m).tolist()) <= {0, 255} and m.shape == (r.size[1], r.size[0])\n                        and int(on.sum()) == e.area and env == e.bbox):\n                    bad.append(f\"{r.record_id}:{s}\")\n        check(\"evidence masks checked\", n, 2752)\n        check(\"masks: values 0/255, image size, area and envelope match the metadata\", len(bad), 0)\n        if bad:\n            print(\"   first offenders:\", bad[:5])\n    else:\n        print(\"[skip] mask checks: segmentation/ is not on disk\")\n\n    print(\"\\nAUDIT\", \"FAILED: \" + \", \".join(FAILED) if FAILED else \"PASSED\")\n    return 1 if FAILED else 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/build_geometry.py": "#!/usr/bin/env python3\n\"\"\"Feasibility of growth controls for the pair supports -> outputs/geometry/feasibility.csv\"\"\"\nfrom __future__ import annotations\n\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import CATEGORY_ORDER, find_release_root, load_release  # noqa: E402\nfrom mfrob.geometry.controls import growth_feasible  # noqa: E402\nfrom mfrob.geometry.supports import SUPPORTS, forbidden_zone, pair_union  # noqa: E402\n\nrows = []\nfor r in load_release(find_release_root()):\n    u = pair_union(r)\n    row = {\"record_id\": r.record_id, \"cat\": r.cat, \"strict_eligible\": r.strict_eligible, \"W\": r.size[0], \"H\": r.size[1]}\n    for name, fn in SUPPORTS.items():\n        s = fn(r)\n        row[f\"n_{name}\"] = int(s.sum())\n        row[f\"growth_{name}\"] = growth_feasible(forbidden_zone(s, u), int(s.sum()))\n    row[\"same_pixels\"] = row[\"n_pair_union\"] == row[\"n_pair_envelope\"] and bool((u == SUPPORTS[\"pair_envelope\"](r)).all())\n    rows.append(row)\nF = pd.DataFrame(rows)\nout = Path(\"outputs/geometry\"); out.mkdir(parents=True, exist_ok=True)\nF.to_csv(out / \"feasibility.csv\", index=False)\nT = F.groupby(\"cat\")[[\"growth_pair_union\", \"growth_pair_envelope\", \"same_pixels\"]].sum().reindex(CATEGORY_ORDER)\nT.loc[\"ALL\"] = T.sum()\nprint(T.astype(int).to_string())\n", "scripts/clean.py": "#!/usr/bin/env python3\n\"\"\"Clean pass of one model over the release.   python scripts/clean.py --model qwen2vl_2b [--limit 10]\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nimport yaml\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nfrom mfrob.data.release import find_release_root, load_release  # noqa: E402\nfrom mfrob.models.qwen_vl import QwenVL  # noqa: E402\nfrom mfrob.run.clean_pass import run_clean_pass  # noqa: E402\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--limit\", type=int, default=None, help=\"first N pairs of every category (smoke test)\")\n    ap.add_argument(\"--out\", type=Path, default=None)\n    ap.add_argument(\"--shard\", default=None, help=\"i/n: take every n-th pair starting from i (one worker per GPU)\")\n    ap.add_argument(\"--categories\", nargs=\"+\", default=None, help=\"release category names to keep\")\n    args = ap.parse_args()\n    cfg = yaml.safe_load(Path(f\"configs/models/{args.model}.yaml\").read_text())\n    records = load_release(find_release_root())\n    if args.categories:\n        records = [r for r in records if r.category in args.categories]\n    if args.limit:\n        seen, keep = {}, []\n        for r in records:\n            if seen.setdefault(r.category, 0) < args.limit:\n                keep.append(r); seen[r.category] += 1\n        records = keep\n    tag = \"\"\n    if args.shard:\n        i, n = (int(v) for v in args.shard.split(\"/\"))\n        records, tag = records[i::n], f\"s{i}-\"\n    out = args.out or Path(\"outputs/logits\") / cfg[\"name\"] / (\"clean_smoke\" if args.limit else \"clean\")\n    model = QwenVL(cfg)\n    yes, no = model.yes_no_ids()\n    meta = {\"model\": model.name, \"hf_id\": model.hf_id, \"revision\": model.revision, \"device\": model.device,\n            \"dtype\": str(model.dtype), \"max_pixels\": model.max_pixels, \"yes_ids\": yes, \"no_ids\": no,\n            \"prompt_template\": model.prompt_template}\n    out.mkdir(parents=True, exist_ok=True)\n    (out / \"run_meta.json\").write_text(json.dumps(meta, ensure_ascii=False, indent=1))\n    print(json.dumps(meta, ensure_ascii=False)[:600])\n    run_clean_pass(model, records, out, tag=tag)\n\n\nif __name__ == \"__main__\":\n    main()\n", "scripts/download_release.py": "#!/usr/bin/env python3\n\"\"\"Download the MechFaithBench v1.9 release from its public Google Drive folder.\n\n`gdown --folder` stops at 50 files per folder, so this script lists folders through the\nembedded folder view (which returns every entry) and downloads files by id. Every file is\nverified against checksums.sha256; files that are already present and correct are skipped.\n\n    python scripts/download_release.py --out data/MechFaithBench              # metadata + images\n    python scripts/download_release.py --out data/MechFaithBench --with-masks # + segmentation masks\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport concurrent.futures as cf\nimport hashlib\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nimport requests\n\nFOLDER_ID = \"1SYLfupQGj_z7cMjau_tbwuZlFhOhinh9\"\nUA = {\"User-Agent\": \"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/126.0 Safari/537.36\"}\nENTRY = re.compile(r'<div class=\"flip-entry\" id=\"entry-([^\"]+)\".*?flip-entry-title\">([^<]*)<', re.S)\nMETA = [\"annotations.csv\", \"roi_annotations.json\", \"checksums.sha256\", \"manifest.json\", \"README.md\", \"MASK_SCHEMA.md\"]\n\n\ndef list_folder(session: requests.Session, folder_id: str) -> dict[str, str]:\n    \"\"\"name -> id for every entry of a public Drive folder.\"\"\"\n    for attempt in range(4):\n        r = session.get(f\"https://drive.google.com/embeddedfolderview?id={folder_id}#list\", timeout=60)\n        if r.status_code == 200:\n            return {name: fid for fid, name in ENTRY.findall(r.text)}\n        time.sleep(2 * (attempt + 1))\n    raise RuntimeError(f\"cannot list folder {folder_id}: HTTP {r.status_code}\")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, \"rb\") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b\"\"):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef fetch(session: requests.Session, file_id: str, dst: Path, expected: str | None) -> str:\n    if dst.exists() and (expected is None or sha256(dst) == expected):\n        return \"kept\"\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    url = f\"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t\"\n    last = \"\"\n    for attempt in range(4):\n        try:\n            r = session.get(url, timeout=120)\n            if r.status_code == 200 and not r.content[:15].lower().startswith(b\"<!doctype html\"):\n                if expected is None or hashlib.sha256(r.content).hexdigest() == expected:\n                    tmp = dst.with_suffix(dst.suffix + \".part\")\n                    tmp.write_bytes(r.content)\n                    tmp.replace(dst)\n                    return \"downloaded\"\n                last = \"checksum mismatch\"\n            else:\n                last = f\"HTTP {r.status_code}\"\n        except requests.RequestException as e:\n            last = str(e)[:80]\n        time.sleep(2 * (attempt + 1))\n    return f\"FAILED ({last})\"\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--out\", type=Path, required=True)\n    ap.add_argument(\"--with-masks\", action=\"store_true\", help=\"also download segmentation/ (3,000 files)\")\n    ap.add_argument(\"--workers\", type=int, default=12)\n    args = ap.parse_args()\n    out: Path = args.out\n    out.mkdir(parents=True, exist_ok=True)\n    session = requests.Session()\n    session.headers.update(UA)\n\n    top = list_folder(session, FOLDER_ID)\n    for name in META:\n        print(f\"{name}: {fetch(session, top[name], out / name, None)}\")\n    checksums = {}\n    for line in (out / \"checksums.sha256\").read_text().splitlines():\n        if line.strip():\n            digest, path = line.split(None, 1)\n            checksums[path.lstrip(\"*\").strip()] = digest\n    for name in META:\n        if name in checksums and sha256(out / name) != checksums[name]:\n            print(f\"checksum mismatch for {name}\", file=sys.stderr)\n            return 1\n\n    jobs: list[tuple[str, Path, str]] = []          # (file id, destination, expected sha256)\n    for sub, sub_id in list_folder(session, top[\"images\"]).items():\n        for name, fid in list_folder(session, sub_id).items():\n            jobs.append((fid, out / \"images\" / sub / name, checksums[f\"images/{sub}/{name}\"]))\n    for name, fid in list_folder(session, top[\"images_repaired\"]).items():\n        jobs.append((fid, out / \"images_repaired\" / name, checksums[f\"images_repaired/{name}\"]))\n    if args.with_masks:\n        seg_dirs = list_folder(session, top[\"segmentation\"])\n        todo = {rid: did for rid, did in seg_dirs.items()\n                if not all((out / p).exists() for p in checksums if p.startswith(f\"segmentation/{rid}/\"))}\n        print(f\"listing {len(todo)} mask folders ...\")\n        with cf.ThreadPoolExecutor(args.workers) as ex:\n            for rid, entries in zip(todo, ex.map(lambda d: list_folder(session, d), todo.values())):\n                for name, fid in entries.items():\n                    jobs.append((fid, out / \"segmentation\" / rid / name, checksums[f\"segmentation/{rid}/{name}\"]))\n\n    print(f\"{len(jobs)} files to check\")\n    t0, stats = time.time(), {}\n    with cf.ThreadPoolExecutor(args.workers) as ex:\n        futures = [ex.submit(fetch, session, fid, dst, exp) for fid, dst, exp in jobs]\n        for i, fut in enumerate(cf.as_completed(futures), 1):\n            res = fut.result()\n            stats[res.split(\" \")[0]] = stats.get(res.split(\" \")[0], 0) + 1\n            if res.startswith(\"FAILED\"):\n                print(res, file=sys.stderr)\n            if i % 400 == 0 or i == len(jobs):\n                print(f\"  {i}/{len(jobs)}  {stats}  {time.time() - t0:.0f}s\", flush=True)\n    return 1 if stats.get(\"FAILED\") else 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/intervene.py": "#!/usr/bin/env python3\n\"\"\"Reference experiment R0 and its variants: target occlusion + K growth controls on pairs with B = 1.\n\n    python scripts/intervene.py --model qwen2vl_2b [--operator black] [--support pair_union] [--shard 0/2] [--limit 5]\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport yaml\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / \"src\"))\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom analyze_clean import pair_table  # noqa: E402\nfrom mfrob.data.release import find_release_root, load_release  # noqa: E402\nfrom mfrob.models.qwen_vl import QwenVL  # noqa: E402\nfrom mfrob.run.clean_pass import load_logits  # noqa: E402\nfrom mfrob.run.intervene import run_interventions  # noqa: E402\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--operator\", default=\"black\")\n    ap.add_argument(\"--support\", default=\"pair_union\")\n    ap.add_argument(\"--k\", type=int, default=4)\n    ap.add_argument(\"--shard\", default=None)\n    ap.add_argument(\"--limit\", type=int, default=None, help=\"first N clean-correct pairs of every category (smoke test)\")\n    args = ap.parse_args()\n    cfg = yaml.safe_load(Path(f\"configs/models/{args.model}.yaml\").read_text())\n    records = load_release(find_release_root())\n    P = pair_table(load_logits(Path(\"outputs/logits\") / cfg[\"name\"] / \"clean\"), records)\n    ok = set(P[P.B].record_id)\n    records = [r for r in records if r.record_id in ok]\n    print(f\"pairs with B = 1: {len(records)} of {len(P)}\")\n    if args.limit:\n        seen, keep = {}, []\n        for r in records:\n            if seen.setdefault(r.category, 0) < args.limit:\n                keep.append(r); seen[r.category] += 1\n        records = keep\n    tag = \"\"\n    if args.shard:\n        i, n = (int(v) for v in args.shard.split(\"/\"))\n        records, tag = records[i::n], f\"s{i}-\"\n    name = f\"{args.operator}__{args.support}__growth_k{args.k}\" + (\"__smoke\" if args.limit else \"\")\n    run_interventions(QwenVL(cfg), records, Path(\"outputs/logits\") / cfg[\"name\"] / name, operator=args.operator,\n                      support=args.support, k=args.k, tag=tag)\n\n\nif __name__ == \"__main__\":\n    main()\n", "configs/data.yaml": "# Where the MechFaithBench v1.9 release lives. The first existing path wins; if none exists,\n# annotations.csv is searched for recursively under the search roots.\nrelease_roots:\n  - data/MechFaithBench\n  - /kaggle/input/datasets/fluloeo/mechfaithbench/MechFaithBench_v1_9_final_2026-07-29\nsearch_roots:\n  - data\n  - /kaggle/input\nsubset_seed: 0\nbootstrap_seed: 20260803\n", "configs/models/qwen2vl_2b.yaml": "name: qwen2vl_2b\nhf_id: Qwen/Qwen2-VL-2B-Instruct\nfamily: qwen_vl\nmax_pixels: 409600        # 640 x 640; the paper does not state its resolution (PLAN.md, stage 2)\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "configs/models/qwen2vl_2b_native.yaml": "name: qwen2vl_2b_native\nhf_id: Qwen/Qwen2-VL-2B-Instruct\nfamily: qwen_vl\nmax_pixels: 12845056      # processor default from the model card: 1024x1024 images stay at native size\ndtype: auto               # cuda: bf16 on Ampere+, fp16 otherwise; mps and cpu: fp32\nbatch_size: 2\n", "pyproject.toml": "[project]\nname = \"mfrob\"\nversion = \"0.1.0\"\ndescription = \"Robustness of the MechFaithBench behavioural protocol\"\nrequires-python = \">=3.11\"\n\n[tool.pytest.ini_options]\npythonpath = [\"src\"]\ntestpaths = [\"tests\"]\n\n[tool.black]\nline-length = 110\n\n[tool.ruff]\nline-length = 110\n"}

for rel, text in FILES.items():
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
os.chdir(REPO)
print(len(FILES), "files written to", REPO)

28 files written to /kaggle/working/mfrob


## Чекпоинты

Все результаты лежат в `outputs/` внутри копии репозитория. Архив обновляется раз в 15 минут и после каждого шага.
Чтобы продолжить в новой сессии, загрузите архив прошлого запуска как датасет и подключите его: ноутбук найдёт его по имени.

In [3]:
_last_zip = time.time()

def make_zip(note=""):
    global _last_zip
    tmp, n = ZIP_PATH + ".tmp", 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, names in os.walk("outputs"):
            for name in names:
                z.write(os.path.join(root, name)); n += 1
    os.replace(tmp, ZIP_PATH); _last_zip = time.time()
    print(f"[checkpoint {note}] {n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)")

def restore():
    if glob.glob("outputs/logits/**/*.parquet", recursive=True):
        print("outputs/ already has results -- continuing from them"); return
    cands = [RESTORE_ZIP] if RESTORE_ZIP else sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(ZIP_PATH)}", recursive=True), key=os.path.getmtime)
    cands = [c for c in cands if c and os.path.exists(c)]
    if not cands:
        print("no previous archive -- starting from scratch"); return
    with zipfile.ZipFile(cands[-1]) as z:
        z.extractall(".")
    print("restored from", cands[-1], "|", len(glob.glob("outputs/logits/**/*.parquet", recursive=True)), "result files")

def run_workers(script_args, name):
    """One process per GPU, each on its own share of the pairs; prints progress and archives periodically."""
    os.makedirs("outputs/logs", exist_ok=True)
    procs = []
    for i in range(N_GPUS):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
        log = open(f"outputs/logs/{name}_gpu{i}.log", "a")
        procs.append((subprocess.Popen([sys.executable, "-u"] + script_args + ["--shard", f"{i}/{N_GPUS}"], stdout=log, stderr=subprocess.STDOUT, env=env), log))
    last = [""] * N_GPUS
    while any(p.poll() is None for p, _ in procs):
        time.sleep(20)
        for i in range(N_GPUS):
            lines = [l for l in open(f"outputs/logs/{name}_gpu{i}.log", errors="ignore").read().splitlines() if " pairs" in l]
            if lines and lines[-1] != last[i]:
                last[i] = lines[-1]; print(f"gpu{i}: {lines[-1].strip()}")
        if time.time() - _last_zip > CHECKPOINT_MINUTES * 60:
            make_zip("periodic")
    for i, (p, log) in enumerate(procs):
        log.close()
        if p.returncode != 0:
            print(f"--- gpu{i} failed, end of its log:\n" + "".join(open(f"outputs/logs/{name}_gpu{i}.log", errors="ignore").readlines()[-25:]))
            raise RuntimeError(f"{name}: worker {i} exited with code {p.returncode}")
    make_zip(name)

def run(script_args):
    r = subprocess.run([sys.executable] + script_args, capture_output=True, text=True)
    print(r.stdout[-6000:]); 
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError(" ".join(script_args))

restore()

no previous archive -- starting from scratch


## 1. Проверка данных

Сверяет релиз с тем, что ожидает код: число пар, категории, размеры, контрольные суммы картинок, маски.

In [4]:
run(["scripts/audit.py"])

release: /kaggle/input/datasets/fluloeo/mechfaithbench/MechFaithBench_v1_9_final_2026-07-29
[ok] unique record ids: 1600
[ok] categories: {'add': 212, 'replace_appearance': 256, 'surface_material_change': 218, 'color_alter': 209, 'counting': 150, 'relation': 110, 'remove': 223, 'textual_change': 222}
[ok] gold (original, edited): {('no', 'yes'): 1376, ('yes', 'no'): 224}
[ok] categories with yes on the original side: {'remove': 223, 'textual_change': 1}
[ok] strict eligible: 1591
[ok] protocols: {'target_only': 1538, 'attribute_specific': 9, 'target_plus_reference': 53}
[ok] target_plus_reference only in relation: {'relation'}
[ok] absent sides: {('add', 'original'): 212, ('counting', 'edited'): 8, ('counting', 'original'): 1, ('remove', 'edited'): 223, ('textual_change', 'edited'): 1, ('textual_change', 'original'): 3}
[ok] no record absent on both sides: True
[ok] absent sides carry no evidence: True
[ok] present sides carry a mask, boxes and an area: True
[ok] csv envelope equals le

## 2. Чистый проход

По два прохода на пару (обе картинки сразу). Модель скачивается один раз до запуска процессов.

In [5]:
import yaml
from huggingface_hub import snapshot_download
snapshot_download(yaml.safe_load(open(f"configs/models/{MODEL}.yaml"))["hf_id"])
clean_args = ["scripts/clean.py", "--model", MODEL, "--out", f"outputs/logits/{MODEL}/clean"] + (["--limit", "3"] if SMOKE else [])
run_workers(clean_args, "clean")
run(["scripts/analyze_clean.py", "--model", MODEL])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

gpu0: clean pass: 800 pairs, 0 already done, 800 to go
gpu1: clean pass: 800 pairs, 0 already done, 800 to go
gpu0: 50/800 pairs | 0.26 s per forward | left ~7 min
gpu1: 50/800 pairs | 0.26 s per forward | left ~7 min
gpu0: 100/800 pairs | 0.25 s per forward | left ~6 min
gpu1: 100/800 pairs | 0.25 s per forward | left ~6 min
gpu0: 150/800 pairs | 0.26 s per forward | left ~6 min
gpu1: 150/800 pairs | 0.26 s per forward | left ~6 min
gpu0: 200/800 pairs | 0.27 s per forward | left ~5 min
gpu1: 200/800 pairs | 0.28 s per forward | left ~6 min
gpu0: 250/800 pairs | 0.28 s per forward | left ~5 min
gpu1: 250/800 pairs | 0.29 s per forward | left ~5 min
gpu0: 300/800 pairs | 0.29 s per forward | left ~5 min
gpu1: 300/800 pairs | 0.30 s per forward | left ~5 min
gpu0: 350/800 pairs | 0.31 s per forward | left ~5 min
gpu1: 350/800 pairs | 0.32 s per forward | left ~5 min
gpu0: 400/800 pairs | 0.32 s per forward | left ~4 min
gpu1: 400/800 pairs | 0.32 s per forward | left ~4 min
gpu0: 450/80

### Проверка разрешения

Все картинки категорий counting и relation — 1024×1024, и в основном прогоне они уменьшаются до 616×616. Остальные категории не уменьшаются.
Здесь те же 260 пар пересчитываются без уменьшения. Если точность в counting заметно вырастет, основную настройку разрешения надо менять.

In [6]:
if RUN_NATIVE_CHECK:
    native = MODEL + "_native"
    run_workers(["scripts/clean.py", "--model", native, "--categories", "counting", "relation", "--out", f"outputs/logits/{native}/clean"] + (["--limit", "3"] if SMOKE else []), "clean_native")
    run(["scripts/analyze_clean.py", "--model", native])

gpu0: clean pass: 130 pairs, 0 already done, 130 to go
gpu1: clean pass: 130 pairs, 0 already done, 130 to go
gpu0: 50/130 pairs | 1.74 s per forward | left ~5 min
gpu1: 50/130 pairs | 1.85 s per forward | left ~5 min
gpu0: 100/130 pairs | 1.74 s per forward | left ~2 min
gpu1: 100/130 pairs | 1.83 s per forward | left ~2 min
gpu0: 130/130 pairs | 1.74 s per forward | left ~0 min
gpu1: 130/130 pairs | 1.83 s per forward | left ~0 min
[checkpoint clean_native] 18 files -> /kaggle/working/mfrob_outputs_qwen2vl_2b.zip (0.4 MB)
            pairs  pair accuracy (PA)  right ordering  yes on both  no on both
cat                                                                           
add           NaN                 NaN             NaN          NaN         NaN
remove        NaN                 NaN             NaN          NaN         NaN
color         NaN                 NaN             NaN          NaN         NaN
counting    150.0               0.633           0.927        0.040       0.

## 3. Закрашивание и контроль

Для каждой верно решённой пары: закрашенная область и `K` контрольных кусков, на обеих картинках. Это 2 × (1 + K) проходов на пару.
Пары, для которых контрольный кусок такой площади не помещается, пропускаются и записываются в `infeasible-*.txt`.

In [7]:
EXPERIMENT = f"{OPERATOR}__{SUPPORT}__growth_k{K}"
run_workers(["scripts/intervene.py", "--model", MODEL, "--operator", OPERATOR, "--support", SUPPORT, "--k", str(K)], "r0")
run(["scripts/analyze_er.py", "--model", MODEL, "--experiment", EXPERIMENT])
if RUN_ENVELOPE:
    run_workers(["scripts/intervene.py", "--model", MODEL, "--operator", OPERATOR, "--support", "pair_envelope", "--k", str(K)], "r0_envelope")
    run(["scripts/analyze_er.py", "--model", MODEL, "--experiment", f"{OPERATOR}__pair_envelope__growth_k{K}"])

gpu0: interventions [black, pair_union, growth, K=4]: 637 pairs, 0 done, 637 to go
gpu1: interventions [black, pair_union, growth, K=4]: 637 pairs, 0 done, 637 to go
gpu0: 25/637 pairs | 0.28 s per forward | left ~29 min
gpu1: 25/637 pairs | 0.31 s per forward | left ~32 min
gpu0: 50/637 pairs | 0.28 s per forward | left ~28 min
gpu1: 50/637 pairs | 0.31 s per forward | left ~31 min
gpu0: 75/637 pairs | 0.28 s per forward | left ~27 min
gpu1: 75/637 pairs | 0.31 s per forward | left ~29 min
gpu0: 100/637 pairs | 0.29 s per forward | left ~26 min
gpu1: 100/637 pairs | 0.31 s per forward | left ~27 min
gpu1: 125/637 pairs | 0.31 s per forward | left ~26 min
gpu0: 150/637 pairs | 0.29 s per forward | left ~23 min
gpu1: 150/637 pairs | 0.32 s per forward | left ~25 min
gpu0: 175/637 pairs | 0.29 s per forward | left ~22 min
gpu1: 175/637 pairs | 0.32 s per forward | left ~24 min
gpu0: 200/637 pairs | 0.30 s per forward | left ~21 min
gpu1: 200/637 pairs | 0.32 s per forward | left ~22 min


## 4. Архив

Как читать таблицу выше: `er` — на сколько закрашивание объекта снижает вероятность правильного ответа сильнее, чем закрашивание контроля
(в среднем по двум картинкам пары); `er_yes` и `er_no` — то же отдельно для картинки с ответом «да» и с ответом «нет»;
`roa` — доля пар, где закрашивание объекта вреднее каждого из контролей.

Скачайте архив по ссылке ниже или запускайте ноутбук через Save Version → Save & Run All, тогда он останется во вкладке Output.

In [8]:
make_zip("final")
print(sorted(glob.glob("outputs/reports/*")))
if IS_KAGGLE:
    from IPython.display import FileLink, display
    os.chdir(BASE); display(FileLink(os.path.basename(ZIP_PATH)))

[checkpoint final] 39 files -> /kaggle/working/mfrob_outputs_qwen2vl_2b.zip (1.5 MB)
['outputs/reports/clean_qwen2vl_2b.json', 'outputs/reports/clean_qwen2vl_2b_native.json', 'outputs/reports/er_qwen2vl_2b_black__pair_union__growth_k4.json']


/kaggle/working/mfrob_outputs_qwen2vl_2b.zip